In [1]:
import os
import sqlite3
import logging
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Import your custom library
# Ensure loan_processor.py is in the same directory as this notebook
try:
    from loan_processor import LoanProcessor
except ImportError:
    print("⚠️ Error: loan_processor.py not found in the current directory.")

# Load environment variables (Ensure .env contains GOOGLE_AI_STUDIO)
load_dotenv()

# --- Configuration ---
# Use raw string (r"...") for Windows paths
ONEDRIVE_ROOT = Path(r"C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Clientes") 
SEARCH_KEYWORD = "credito ordinario"
DB_PATH = "processing_history.db"
OUTPUT_JSON_FOLDER = "client_json_data"

# The LoanProcessor library configures logging on import, 
# but we can add a handler to print to the notebook output if desired.
logging.getLogger().setLevel(logging.INFO)


In [2]:
def init_db():
    """Initialize the tracking database if it doesn't exist."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    # We track file path, modification time, processed date, and status
    c.execute('''CREATE TABLE IF NOT EXISTS file_history
                 (file_path TEXT PRIMARY KEY, 
                  last_modified REAL, 
                  processed_at TEXT, 
                  status TEXT)''')
    conn.commit()
    return conn

def should_process(conn, file_path):
    """
    Decides if a file needs processing.
    Returns True if file is new, modified since last run, or previously failed.
    """
    try:
        current_mtime = file_path.stat().st_mtime
        c = conn.cursor()
        c.execute("SELECT last_modified, status FROM file_history WHERE file_path=?", (str(file_path),))
        row = c.fetchone()
        
        if row is None:
            return True # New file
            
        last_recorded_mtime, status = row
        
        # Process if file was modified since last record, or previous attempt failed
        if current_mtime > last_recorded_mtime or status != 'SUCCESS':
            return True
            
        return False # Already processed and unchanged
    except FileNotFoundError:
        return False

def mark_completed(conn, file_path, status="SUCCESS"):
    """Updates the database record for a file."""
    try:
        current_mtime = file_path.stat().st_mtime
        c = conn.cursor()
        c.execute('''INSERT OR REPLACE INTO file_history 
                     (file_path, last_modified, processed_at, status) 
                     VALUES (?, ?, ?, ?)''', 
                  (str(file_path), current_mtime, datetime.now().isoformat(), status))
        conn.commit()
    except Exception as e:
        print(f"⚠️ Error updating DB for {file_path.name}: {e}")


In [3]:
def find_target_files(root_path, keyword, silent=False):
    """
    Generator that yields paths to PDF files containing the keyword.
    """
    root = Path(root_path)
    if not root.exists():
        print(f"❌ Error: Path not found: {root}")
        return

    if not silent:
        print(f"📂 Scanning {root}...")
    
    # rglob recursively finds files. We filter for .pdf and the keyword.
    for path in root.rglob("*.pdf"):
        if keyword.lower() in path.name.lower():
            yield path



In [4]:
def count_pending_files(conn, root_path, keyword):
    """
    Helper function to count how many files actually need processing
    before starting the main loop.
    """
    print("🔍 Analyzing workload (counting pending files)...")
    count = 0
    # We use silent=True to avoid printing "Scanning..." twice
    for file_path in find_target_files(root_path, keyword, silent=True):
        if should_process(conn, file_path):
            count += 1
    return count

In [5]:
conn = init_db()
count_pending_files(conn, ONEDRIVE_ROOT, SEARCH_KEYWORD)

🔍 Analyzing workload (counting pending files)...


77

In [6]:
def run_batch_process():
    # 1. Initialize the LoanProcessor
    try:
        processor = LoanProcessor() 
        print("✅ LoanProcessor initialized successfully.")
    except Exception as e:
        print(f"❌ Failed to initialize LoanProcessor: {e}")
        return

    conn = init_db()
    
    try:
        # 2. Count files first
        total_pending = count_pending_files(conn, ONEDRIVE_ROOT, SEARCH_KEYWORD)
        
        if total_pending == 0:
            print("✅ No new files to process. Everything is up to date!")
            return

        print(f"📊 Found {total_pending} files waiting to be processed.")
        print(f"🚀 Starting batch process...")
        print(f"📂 Output folder: {Path(OUTPUT_JSON_FOLDER).absolute()}")
        
        processed_count = 0
        errors_count = 0
        
        # 3. Iterate through files found by the scanner
        for file_path in find_target_files(ONEDRIVE_ROOT, SEARCH_KEYWORD):
            
            if should_process(conn, file_path):
                processed_count += 1
                print(f"⚡ Processing [{processed_count}/{total_pending}]: {file_path.name}")
                
                # --- CALL YOUR LIBRARY HERE ---
                result = processor.process_single_file(
                    pdf_path=str(file_path), 
                    output_folder=OUTPUT_JSON_FOLDER
                )
                # ------------------------------
                
                if result:
                    print(f"   ✅ Extracted data.")
                    mark_completed(conn, file_path, status="SUCCESS")
                else:
                    print(f"   ❌ Extraction failed (check logs).")
                    mark_completed(conn, file_path, status="ERROR")
                    errors_count += 1
                
    except KeyboardInterrupt:
        print("\n🛑 Process interrupted by user.")
    finally:
        conn.close()
        
    print("\n" + "="*30)
    print(f"✅ Batch Complete")
    print(f"🆕 Processed: {processed_count}/{total_pending}")
    print(f"❌ Errors:    {errors_count}")
    print("="*30)

# Execute
run_batch_process()


✅ LoanProcessor initialized successfully.
🔍 Analyzing workload (counting pending files)...
📊 Found 77 files waiting to be processed.
🚀 Starting batch process...
📂 Output folder: c:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\BONANZA WEB\bonanza-customers\client_json_data
📂 Scanning C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Clientes...


2026-03-02 20:25:26 - INFO - Processing: C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Clientes\Vargas Gutierrez, Diana Patricia 1\20180731 Credito ordinario Vargas Gutierrez, Diana Patricia 1.pdf


⚡ Processing [1/77]: 20180731 Credito ordinario Vargas Gutierrez, Diana Patricia 1.pdf


2026-03-02 20:25:27 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
2026-03-02 20:25:39 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWyZGkIJBwFpTR7VQ3KCkP_I5h0pzdacXRXqHXE0ifge8JnFhHbkUdFRJFTvdmhq_kb7-g778mGEZu23jERAAKzgoQIZreY4TnskmBHPrfo&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-02 20:25:39 - INFO - AFC is enabled with max remote calls: 10.
2026-03-02 20:25:57 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-03-02 20:25:57 - INFO - Saved: client_json_data\20180731 Credito ordinario Vargas Gutierrez, Diana Patricia 1_extracted.json
2026-03-02 20:25:58 - INFO - HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/prvgw4oxzql2 "HTTP/1.1 200 OK"
2026-03-02 20:25:58 - INFO - Processing: C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Client

   ✅ Extracted data.
⚡ Processing [2/77]: 20220801 Credito ordinario Vargas Gutierrez, Diana Patricia 1.pdf


...

   ✅ Extracted data.
⚡ Processing [76/77]: 20250318 Credito ordinario Zuñiga Ortiz, Sonia Alejandra 1.pdf


2026-03-02 21:13:43 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
2026-03-02 21:13:55 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWxgIsdoJLHlnvG1qBx8m_XNaO09a57zuHFq6ADdOyiq5NeDp5nMUXsPduaYKNEC0Fineq2D2Y-uLfHvXpyYTLXHhOeUw383SzxjyU68TDQ&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-02 21:13:55 - INFO - AFC is enabled with max remote calls: 10.
2026-03-02 21:14:15 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-03-02 21:14:15 - INFO - Saved: client_json_data\20250318 Credito ordinario Zuñiga Ortiz, Sonia Alejandra 1_extracted.json
2026-03-02 21:14:15 - INFO - HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/h5h1x6scvy0e "HTTP/1.1 200 OK"
2026-03-02 21:14:15 - INFO - Processing: C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Clientes\

   ✅ Extracted data.
⚡ Processing [77/77]: 20220924 Credito ordinario Zuñiga Salazar, Javier Camilo 1.pdf


2026-03-02 21:14:17 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files "HTTP/1.1 200 OK"
2026-03-02 21:14:28 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/upload/v1beta/files?upload_id=AGQBYWyRXGFdnZVrKl0x7gcL5WclJudxpaYdAO-eSjpjimIPTkSRmTspgnlCztE7-CVQhaNs-FokgsSH2Jr7PE92WUxaPKbM6qvG6krBf54CpDw&upload_protocol=resumable "HTTP/1.1 200 OK"
2026-03-02 21:14:28 - INFO - AFC is enabled with max remote calls: 10.
2026-03-02 21:14:41 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-03-02 21:14:41 - INFO - Saved: client_json_data\20220924 Credito ordinario Zuñiga Salazar, Javier Camilo 1_extracted.json
2026-03-02 21:14:41 - INFO - HTTP Request: DELETE https://generativelanguage.googleapis.com/v1beta/files/8ca5zoqryod1 "HTTP/1.1 200 OK"


   ✅ Extracted data.

✅ Batch Complete
🆕 Processed: 77/77
❌ Errors:    0
